# Embedding large data sets

Embedding large data sets typically requires more care. Using various tricks described in *preserving_global_structure* can become quite slow to run. Instead, we can take a smaller, manageable sample of our data set, obtain a good visualization of that. Then, we can add the remaining points to the embedding and use that as our initialization.

Remember that the initialization largely affects the structure of the embedding. This way, our initialization provides the global structure for the embedding, and the subsequent optimization can focus on preserving local strucutre.

In [1]:
import gzip
import pickle

import numpy as np
import openTSNE
from examples import utils

import matplotlib.pyplot as plt
%matplotlib inline

## Load data

The preprocessed data set can be downloaded from http://file.biolab.si/opentsne/benchmark/10x_mouse_zheng.pkl.gz.

In [ ]:
%%time
with gzip.open("data/10x_mouse_zheng.pkl.gz", "rb") as f:
    data = pickle.load(f)

x = data["pca_50"]
y = data["CellType1"]

In [ ]:
print("Data set contains %d samples with %d features" % x.shape)

In [4]:
def plot(x, y, **kwargs):
    fig, ax = plt.subplots(ncols=2, figsize=(16, 8))
    alpha = kwargs.pop("alpha", 0.1)
    utils.plot(
        x,
        np.zeros_like(y),
        ax=ax[0],
        colors={0: "k"},
        alpha=alpha,
        draw_legend=False,
        **kwargs,
    )
    utils.plot(
        x,
        y,
        ax=ax[1],
        colors=utils.MOUSE_10X_COLORS,
        alpha=alpha,
        draw_legend=False,
        **kwargs,
    )

In [5]:
def rotate(degrees):
    phi = degrees * np.pi / 180
    return np.array([
        [np.cos(phi), -np.sin(phi)],
        [np.sin(phi), np.cos(phi)],
    ])

In [ ]:
plot(x, y)

We'll also precompute the full affinities, since we'll be needing it in several places throughout the notebook, and can take a long time to run.

In [ ]:
%%time
aff50 = openTSNE.affinity.PerplexityBasedNN(
    x,
    perplexity=50,
    n_jobs=32,
    random_state=0,
)

In [ ]:
%%time
aff500 = openTSNE.affinity.PerplexityBasedNN(
    x,
    perplexity=500,
    n_jobs=32,
    random_state=0,
)

## Standard t-SNE

First, let's see what standard t-SNE does.

In [9]:
# Because we're given the data representation as the top 50 principal components
# we can just use the top 2 components as the initilization. There is no sense in
# calculating PCA on a PCA representation
init = openTSNE.initialization.rescale(x[:, :2])

In [ ]:
%%time
embedding_standard = openTSNE.TSNE(
    n_jobs=32,
    verbose=True,
).fit(affinities=aff50, initialization=init)

In [ ]:
plot(embedding_standard, y)

This doesn't look too great. The cluster separation is quite poor and the visualization is visually not very appealing.

## Using larger exaggeration

Exaggeration can be used in order to get better separation between clusters. Let's see if that helps.

In [ ]:
%%time
embedding_exag = openTSNE.TSNE(
    exaggeration=4,
    n_jobs=32,
    verbose=True,
).fit(affinities=aff50, initialization=init)

In [ ]:
plot(embedding_exag, y)

The separation has improved quite a bit, but many clusters are still intertwined with others.

## Using a larger perplexity

In [ ]:
%%time
embedding_aff500 = openTSNE.TSNE(
    n_jobs=32,
    verbose=True,
).fit(affinities=aff500, initialization=init)

In [ ]:
plot(embedding_aff500, y)

## ... with higher exaggeration

In [ ]:
%%time
embedding_aff500_exag4 = openTSNE.TSNE(
    exaggeration=4,
    n_jobs=32,
    verbose=True,
).fit(affinities=aff500, initialization=init)

In [ ]:
plot(embedding_aff500_exag4, y)

## Initialize via downsampling

We now perform the sample-transform trick we described above.

### Create train/test split

In [18]:
np.random.seed(0)

In [19]:
indices = np.random.permutation(list(range(x.shape[0])))
reverse = np.argsort(indices)

x_sample, x_rest = x[indices[:25000]], x[indices[25000:]]
y_sample, y_rest = y[indices[:25000]], y[indices[25000:]]

### Create sample embedding

In [ ]:
%%time
sample_affinities = openTSNE.affinity.PerplexityBasedNN(
    x_sample,
    perplexity=500,
    n_jobs=32,
    random_state=0,
    verbose=True,
)

In [ ]:
%time sample_init = openTSNE.initialization.pca(x_sample, random_state=42)

In [ ]:
%time sample_embedding = openTSNE.TSNE(n_jobs=32, verbose=True).fit(affinities=sample_affinities, initialization=sample_init)

In [ ]:
plot(sample_embedding, y[indices[:25000]], alpha=0.5)

### Learn the full embedding

In [ ]:
%time rest_init = sample_embedding.prepare_partial(x_rest, k=1, perplexity=1/3)

In [25]:
init_full = np.vstack((sample_embedding, rest_init))[reverse]

In [ ]:
plot(init_full, y)

In [ ]:
init_full = init_full / (np.std(init_full[:, 0]) * 10000)
np.std(init_full, axis=0)

In [28]:
embedding = openTSNE.TSNEEmbedding(
    init_full,
    aff50,
    n_jobs=32,
    verbose=True,
    random_state=42,
)

In [ ]:
%time embedding1 = embedding.optimize(n_iter=500, exaggeration=12)

In [ ]:
plot(embedding1, y)

In [ ]:
%time embedding2 = embedding1.optimize(n_iter=250, exaggeration=4)

In [ ]:
plot(embedding2, y)

In [ ]:
%time embedding3 = embedding2.optimize(n_iter=250, exaggeration=4)

In [ ]:
plot(embedding3, y)

In [ ]:
%time embedding4 = embedding3.optimize(n_iter=250, exaggeration=4)

In [ ]:
plot(embedding4, y)

## Comparison to UMAP

In [ ]:
from umap import UMAP

In [38]:
umap = UMAP(n_neighbors=15, min_dist=0.1, random_state=1)

In [ ]:
%time embedding_umap = umap.fit_transform(x)

In [ ]:
plot(embedding_umap, y)